<a href="https://colab.research.google.com/github/afif-af/Natural-Language-Processing/blob/main/Hands_On_Project_Sentiment_Analysis_with_LSTM_in_Pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#.


In [44]:
import collections
from collections import Counter
import datasets
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import tqdm

In [45]:
train_data, test_data=datasets.load_dataset("imdb", split=['train', 'test'])



In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

x         = tokenizer("I love AI")
print(x)

{'input_ids': [101, 1045, 2293, 9932, 102], 'token_type_ids': [0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1]}


In [ ]:
def tokenize_example(example, tokenizer, max_length):

    tokenized_output = tokenizer(
        example['text'],
        truncation=True,
        max_length=max_length,
        padding="max_length"
    )

    return {
        "tokens": tokenized_output["input_ids"],
        "attention_mask": tokenized_output["attention_mask"],
        "length": len(tokenized_output["input_ids"])
    }

In [ ]:
train_data[0]

{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far be

In [ ]:
max_length = 256

train_data = train_data.map(
    tokenize_example,
    fn_kwargs={
        "tokenizer": tokenizer,
        "max_length": max_length
    }
)

test_data = test_data.map(
    tokenize_example,
    fn_kwargs={
        "tokenizer": tokenizer,
        "max_length": max_length
    }
)

print(train_data[0])

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far be

In [ ]:
train_data

Dataset({
    features: ['text', 'label', 'tokens', 'attention_mask', 'length'],
    num_rows: 25000
})

In [ ]:
test_data

Dataset({
    features: ['text', 'label', 'tokens', 'attention_mask', 'length'],
    num_rows: 25000
})

In [ ]:
test_size =0.25
train_valid_data =train_data.train_test_split(test_size=test_size)
train_data =train_valid_data["train"]
valid_data =train_valid_data["test"]

In [ ]:
train_data

Dataset({
    features: ['text', 'label', 'tokens', 'attention_mask', 'length'],
    num_rows: 18750
})

In [ ]:
valid_data

Dataset({
    features: ['text', 'label', 'tokens', 'attention_mask', 'length'],
    num_rows: 6250
})

In [ ]:
test_data

Dataset({
    features: ['text', 'label', 'tokens', 'attention_mask', 'length'],
    num_rows: 25000
})

In [ ]:
min_freq = 3
special_tokens = ["<unk>", "<pad>"]
counter = Counter()

for tokens in train_data["tokens"]:
    counter.update(tokens)

# token -> index
vocab = {}

# Add special tokens
for token in special_tokens:
    vocab[token] = len(vocab)

# Add normal tokens
for token, freq in counter.items():

    if freq >= min_freq:
        vocab[token] = len(vocab)


In [ ]:
unk_index =vocab["<unk>"]
pad_index =vocab["<pad>"]

In [ ]:
# Numericalization
def numericalize(example):
    return {
        "input_ids": [
            vocab.get(token, unk_index)
            for token in example["tokens"]
        ]
    }

# Example
sample_tokens = train_data[0]["tokens"]
sample_ids    = numericalize(train_data[0])["input_ids"]

print(sample_tokens[:20])
print(sample_ids[:20])

print("Vocabulary size:", len(vocab))

[101, 1045, 2387, 1996, 2184, 2361, 1012, 1049, 1012, 4760, 1998, 1045, 2442, 2360, 2008, 2023, 3185, 2001, 2498, 2569]
[2, 3, 4, 5, 6, 7, 8, 9, 8, 10, 11, 3, 12, 13, 14, 15, 16, 17, 18, 19]
Vocabulary size: 20286


In [ ]:
train_data =train_data.map(numericalize)

valid_data =valid_data.map(numericalize)
test_data  =test_data.map(numericalize)

Map:   0%|          | 0/18750 [00:00<?, ? examples/s]

Map:   0%|          | 0/6250 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

In [ ]:
train_data =train_data.with_format(type="torch", columns =['input_ids', 'label', "length"])
valid_data =valid_data.with_format(type="torch", columns =["input_ids", 'label', "length"])
test_data  =test_data.with_format(type="torch", columns =["input_ids", "label", "length"])

In [ ]:
train_data

Dataset({
    features: ['text', 'label', 'tokens', 'attention_mask', 'length', 'input_ids'],
    num_rows: 18750
})

In [ ]:
valid_data

Dataset({
    features: ['text', 'label', 'tokens', 'attention_mask', 'length', 'input_ids'],
    num_rows: 6250
})

In [ ]:
test_data

Dataset({
    features: ['text', 'label', 'tokens', 'attention_mask', 'length', 'input_ids'],
    num_rows: 25000
})

In [ ]:
train_data[0]

{'label': tensor(0),
 'length': tensor(256),
 'input_ids': tensor([  2,   3,   4,   5,   6,   7,   8,   9,   8,  10,  11,   3,  12,  13,
          14,  15,  16,  17,  18,  19,   8,  20,   3,  21,  22,  23,   5,  24,
          25,  26,  27,  28,  29,  30,   3,  31,  32,  33,  34,  35,  36,  37,
          38,  39,   3,  40,  32,  33,  41,  42,  43,  44,  45,  46,  47,   8,
          48,  49,  50,  51,  48,  49,  50,  51,  52,  53,  54,  55,  56,  11,
          57,  14,  58,  59,  60,  61,  62,  63,   5,  64,  65,  66,  67,  68,
          69,  70,  71,  66,   5,  72,  29,  73,  74,  75,  76,  77,  30,  15,
          64,  78,  79,  80,  39,  81,  82,   8,   5,  70,  71,  65,  83,  68,
          84,  11,  85,  86,  30,  87,   8,   3,  88,  89,  90,  91,  66,  92,
          93,  94,   8,  13,  95,  96,  97,  61,  62,  98,  32,  99, 100, 101,
          61, 102,  32,  99,  54, 103, 104,  68, 105, 106,  36, 107, 108,  35,
         109,   8,   5, 110, 111, 112, 113,  68,   5,  64,  17,   5, 114,

#Data Loaders

In [ ]:
def get_collate_fn(pad_index):
    def collate_fn(batch):
        # ids
        batch_ids = [
            torch.tensor(i["input_ids"])
            for i in batch
        ]
        batch_ids = nn.utils.rnn.pad_sequence(
            batch_ids,
            padding_value =pad_index,
            batch_first   =True
        )

        # length
        batch_length = torch.tensor(
            [i["length"] for i in batch]
        )
        # label
        batch_label = torch.tensor(
            [i["label"] for i in batch]
        )

        batch = {
            "input_ids": batch_ids,
            "length": batch_length,
            "label": batch_label
        }

        return batch
    return collate_fn

In [ ]:
def get_data_loader(dataset, batch_size, pad_index, shuffle=False):
  collate_fn=get_collate_fn(pad_index)
  data_loader =torch.utils.data.DataLoader(
      dataset    =dataset,
      batch_size =batch_size,
      collate_fn =collate_fn,
      shuffle    =shuffle,
   )
  return data_loader

In [ ]:
batch_size =512
train_data_loader =get_data_loader(train_data, batch_size, pad_index, shuffle=True)
valid_data_loader =get_data_loader(valid_data, batch_size, pad_index )
test_data_loader =get_data_loader(test_data, batch_size, pad_index)


In [ ]:
len(vocab)

20286

In [ ]:
pad_index

1

In [ ]:
vocab

{'<unk>': 0,
 '<pad>': 1,
 101: 2,
 1045: 3,
 2387: 4,
 1996: 5,
 2184: 6,
 2361: 7,
 1012: 8,
 1049: 9,
 4760: 10,
 1998: 11,
 2442: 12,
 2360: 13,
 2008: 14,
 2023: 15,
 3185: 16,
 2001: 17,
 2498: 18,
 2569: 19,
 2348: 20,
 2106: 21,
 2025: 22,
 2681: 23,
 4258: 24,
 5782: 25,
 2026: 26,
 2051: 27,
 2067: 28,
 1006: 29,
 2004: 30,
 2123: 31,
 1005: 32,
 1056: 33,
 2941: 34,
 3477: 35,
 2005: 36,
 5691: 37,
 4902: 38,
 1007: 39,
 2134: 40,
 2428: 41,
 2424: 42,
 2151: 43,
 2417: 44,
 21564: 45,
 2075: 46,
 11647: 47,
 1026: 48,
 7987: 49,
 1013: 50,
 1028: 51,
 2045: 52,
 2020: 53,
 1037: 54,
 2261: 55,
 3210: 56,
 2107: 57,
 2081: 58,
 2033: 59,
 15375: 60,
 1010: 61,
 2021: 62,
 3262: 63,
 2143: 64,
 2790: 65,
 2000: 66,
 8676: 67,
 1997: 68,
 25883: 69,
 5470: 70,
 2326: 71,
 3920: 72,
 1999: 73,
 2568: 74,
 2062: 75,
 2084: 76,
 2287: 77,
 2003: 78,
 6758: 79,
 1054: 80,
 3287: 81,
 4378: 82,
 2041: 83,
 2173: 84,
 2738: 85,
 25012: 86,
 2092: 87,
 2113: 88,
 2017: 89,
 2035: 90,

In [ ]:
train_data

Dataset({
    features: ['text', 'label', 'tokens', 'attention_mask', 'length', 'input_ids'],
    num_rows: 18750
})

In [ ]:
train_data.unique("label")

[0, 1]

In [ ]:
len(train_data.unique("label"))

2

# Model LSTM

In [ ]:
class LSTM(nn.Module):
  def __init__(self, vocab_size, embedding_dim, hidden_dim,output_dim, n_layers, bidirectional, dropout_rate, pad_index):
    super().__init__()
    self.embedding =nn.Embedding(vocab_size, embedding_dim, padding_idx =pad_index)
    self.lstm     =nn.LSTM(embedding_dim, hidden_dim,n_layers, bidirectional=bidirectional, dropout=dropout_rate, batch_first =True )
    self.fc       =nn.Linear(hidden_dim *2 if bidirectional else hidden_dim, output_dim)
    self.dropout  =nn.Dropout(dropout_rate)

  def forward(self, ids, length):
    #ids    =[batch_size, seq_len]
    #length =[batch_size]

    embedded          =self.dropout(self.embedding(ids))
    #embedded =batch_size, seq len, embedding dim
    packed_embedded   =nn.utils.rnn.pack_padded_sequence(
        embedded,
        length,
        batch_first    =True,
        enforce_sorted =False
    )
    packed_output, (hidden,cell) =self.lstm(packed_embedded)
    # hidden =[n_layer, * directions, batch_size, hidden_dim]
    # cell   =[n_layer, * directions, batch_size, hidden_dim]
    output, output_legnth =nn.utils.rnn.pad_packed_sequence(packed_output, )
    # output =[batch size seq len, hidden dim * directions]
    if self.lstm.bidirectional:
      hidden =self.dropout(torch.cat((hidden[-2], hidden[-1]), dim=1))
      #hidden [batch size, hidden dim *2]

    else:
      hidden =self.dropout(hidden[-1])

    prediction =self.fc(hidden)
    return prediction

In [ ]:
vocab_size    = len(vocab)
embedding_dim = 300
hidden_dim    = 300
n_layers      = 2
bidirectional = True
dropout_rate  = 0.5
output_dim    = len(train_data.unique("label"))
model          =LSTM(vocab_size,
                     embedding_dim,
                     hidden_dim,
                     output_dim,
                     n_layers,
                     bidirectional,
                     dropout_rate,
                     pad_index)



In [ ]:
model

LSTM(
  (embedding): Embedding(20286, 300, padding_idx=1)
  (lstm): LSTM(300, 300, num_layers=2, batch_first=True, dropout=0.5, bidirectional=True)
  (fc): Linear(in_features=600, out_features=2, bias=True)
  (dropout): Dropout(p=0.5, inplace=False)
)

In [ ]:
def count_parameters(model):
  return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"The model has {count_parameters(model):,} trainable parameters")

The model has 9,696,602 trainable parameters


In [ ]:
def initialize_weights(m):
  if isinstance(m, nn.Linear):
    nn.init.xavier_normal(m.weight)
    nn.init.zeros_(m.bias)

  elif isinstance(m, nn.LSTM):
    for name, param in m.named_parameters():
      if "bias" in name:
        nn.init.zeros_(param)
      elif "weight" in name:
        nn.init.orthogonal_(param)


In [ ]:
model.apply(initialize_weights)

/tmp/ipykernel_5895/3355809102.py:3: FutureWarning: `nn.init.xavier_normal` is now deprecated in favor of `nn.init.xavier_normal_`.
  nn.init.xavier_normal(m.weight)


LSTM(
  (embedding): Embedding(20286, 300, padding_idx=1)
  (lstm): LSTM(300, 300, num_layers=2, batch_first=True, dropout=0.5, bidirectional=True)
  (fc): Linear(in_features=600, out_features=2, bias=True)
  (dropout): Dropout(p=0.5, inplace=False)
)

In [ ]:
!pip install gensim

In [ ]:
import gensim.downloader as api

# Load pretrained glove vectors
glove = api.load("glove-wiki-gigaword-300")

# word vector
vector = glove["hello"]

print(vector.shape)
print(vector[:10])

(300,)
[-0.33712   -0.21691   -0.0066365 -0.41625   -1.2555    -0.028466
 -0.72195   -0.52887    0.0072085  0.31997  ]


In [ ]:
import gensim.downloader as api
import numpy as np

glove = api.load("glove-wiki-gigaword-300")

pretrained_embedding = torch.tensor(
    np.array([
        glove[word] if word in glove
        else np.random.randn(embedding_dim) # Use embedding_dim instead of hardcoded 100
        for word in vocab.keys() # Iterate over keys instead of calling get_itos()
    ]),
    dtype=torch.float32
)

model.embedding.weight.data = pretrained_embedding

In [ ]:
lr =5e-4
optimizer =optim.Adam(model.parameters(), lr=lr)


In [ ]:
criterion =nn.CrossEntropyLoss()
device    =torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
model     =model.to(device)
criterion =criterion.to(device)

In [ ]:
def train(dataloader, model, criterion, optimizer,device):
  model.train()
  epoch_losses =[]
  epoch_accs   =[]
  for batch in tqdm.tqdm(dataloader, desc ="training..."):
    ids    =batch["input_ids"].to(device)
    length =batch["length"]
    label  =batch["label"].to(device)
    prediction =model(ids, length)
    loss       =criterion (prediction, label)
    accuracy   =get_accuracy(prediction, label)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    epoch_losses.append(loss.item())
    epoch_accs.append(accuracy.item())

  return np.mean(epoch_losses), np.mean(epoch_accs)

In [ ]:
def evaluate(dataloader, model, criterion, device):
  model.eval()
  epoch_losses=[]
  epoch_accs  =[]
  with torch.no_grad():
    for batch in tqdm.tqdm(dataloader, desc ="evaluating..."):
      ids    =batch["input_ids"].to(device)
      length =batch["length"]
      label  =batch["label"].to(device)
      prediction =model(ids, length)
      loss       =criterion (prediction, label)
      accuracy   =get_accuracy(prediction, label)
      # The optimizer.zero_grad(), loss.backward(), and optimizer.step() calls are not needed in evaluation mode
      epoch_losses.append(loss.item())
      epoch_accs.append(accuracy.item())

  return np.mean(epoch_losses), np.mean(epoch_accs)

In [ ]:
def get_accuracy(prediction, label):
  batch_size, _       =prediction.shape
  predicted_classes   =prediction.argmax(dim =-1)
  correct_predictions =predicted_classes.eq(label).sum()
  accuracy            =correct_predictions / batch_size
  return accuracy


In [ ]:
n_epochs =3
best_valid_loss =float("inf")

metrics =collections.defaultdict(list)
for epoch in range(n_epochs):
  train_loss, train_acc =train(
      train_data_loader, model, criterion, optimizer, device

  )
  valid_loss, valid_acc =evaluate(valid_data_loader, model, criterion, device)
  metrics ["train_losses"].append(train_loss)
  metrics ["train_accs"].append(train_acc)
  metrics ["valid_losses"].append(valid_loss)
  metrics ["valid_accs"].append(valid_acc)
  if valid_loss <best_valid_loss:
    best_valid_loss =valid_loss
    torch.save(model.state_dict(), "lstm.pt")
  print(f"epoch: {epoch}")
  print(f"train_loss: {train_loss: .3f}, train_acc: {train_acc:.3f}")
  print(f"valid_loss: {valid_loss: .3f}, valid_acc: {valid_acc:.3f}")


In [ ]:
fig =plt.figure(figsize =(10, 6))
ax =fig.add_subplot(1,1,1)
ax.plot(metrics["train_losses"], label="tran loss")
ax.plot(metrics["valid_losses"], label ="valid loss")
ax.set_xlabel("epoch")
ax.set_ylabel("loss")
ax.set_xticks(range(n_epochs))
ax.legend()
ax.grid()

In [ ]:
fig =plt.figure(figsize =(10, 6))
ax =fig.add_subplot(1,1,1)
ax.plot(metrics["train_accs"], label="train accuracy")
ax.plot(metrics["valid_accs"], label ="valid accuracy")
ax.set_xlabel("epoch")
ax.set_ylabel("accuracy")
ax.set_xticks(range(n_epochs))
ax.legend()
ax.grid()

In [ ]:
model.load_state_dict(torch.load("lstm.pt"))
test_loss, test_acc =evaluate(test_data_loader, model, criterion, device)


In [ ]:
print(f"test_loss {test_loss:.3f}, test_acc: {test_acc:.3f}")

In [ ]:
def predict_sentiment(text, model, tokenizer, vocab, device):
  # Tokenize the input text using the BERT tokenizer
  tokenized_output = tokenizer(text, truncation=True, max_length=max_length) # Use max_length from notebook

  # Create a dummy example dictionary to use with the numericalize function
  dummy_example = {"tokens": tokenized_output["input_ids"]}

  # Numericalize these BERT input_ids using our custom vocab dictionary
  numericalized_result = numericalize(dummy_example)
  numericalized_input_ids = numericalized_result["input_ids"]

  # Calculate length of the numericalized input_ids
  length = torch.LongTensor([len(numericalized_input_ids)])

  # Convert numericalized_input_ids to a PyTorch tensor and add a batch dimension
  tensor = torch.LongTensor(numericalized_input_ids).unsqueeze(dim=0).to(device)

  # Get prediction from the model
  prediction = model(tensor, length).squeeze(dim=0)

  # Calculate probabilities and predicted class
  probability = torch.softmax(prediction, dim=-1)
  predicted_class = prediction.argmax(dim=-1).item()
  predicted_probability = probability[predicted_class].item()

  return predicted_class, predicted_probability

In [ ]:
text ="This fillm id great!"
predict_sentiment (text, model, tokenizer, vocab, device)


In [ ]:
text ="thi film is not terribel , its great!"
predict_sentiment (text, model, tokenizer, vocab, device)